# 0.4 — Score a response: $\log P(\text{response} \mid \text{prompt})$

**Goal.** The core operation of the whole project. Given a prompt and a response, run one forward pass
over `prompt + response` (teacher forcing), take the log-softmax at each position, and sum the
log-probabilities of the *response* tokens only. Report total nats and nats per token.

We build it by hand here, cell by cell, then check it two ways: against the scores `generate` itself
reports while decoding, and against the packaged version in `src/persona_selection/scoring.py`
that later notebooks import.

**The gotcha the README flags:** tokenize prompt and response *together* and mask the prompt. If you
tokenize them separately and concatenate the ids, the tokens at the boundary can differ from what the
model would see in the joint string, and the score silently changes. We demonstrate this below.

In [1]:
import os, sys, time, json
# HF env vars must be set BEFORE transformers is imported (read at import time); env.sh / the persona-ml kernel also set them.
os.environ.setdefault("HF_HOME", "/global/cfs/cdirs/m2612/ozamram/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.prompts import make_prompt

RESULTS = REPO / "results" / "phase0"; RESULTS.mkdir(parents=True, exist_ok=True)

CONFIG = {"model": "Qwen/Qwen2.5-7B", "seed": 0, "question": "What should I do if I find a lost wallet?"}
torch.manual_seed(CONFIG["seed"])
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"])
model = AutoModelForCausalLM.from_pretrained(CONFIG["model"], dtype=torch.bfloat16, device_map="cuda").eval()

prompt = make_prompt(CONFIG["question"])

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## A reference response, with the model's own log-probs

Generate 30 tokens greedily and ask `generate` to keep the per-step scores. `compute_transition_scores`
turns those into the log-probability the model assigned to each token it emitted. This is our ground
truth: the teacher-forced score of exactly this text must reproduce these numbers.

In [2]:
enc = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    gen = model.generate(**enc, max_new_tokens=30, do_sample=False, output_scores=True,
                         return_dict_in_generate=True, pad_token_id=tokenizer.pad_token_id)
gen_ids = gen.sequences[0, enc["input_ids"].shape[1]:]
ref_logprobs = model.compute_transition_scores(gen.sequences, gen.scores, normalize_logits=True)[0]
response = tokenizer.decode(gen_ids)          # no eos appears within 30 greedy tokens for this question
print(repr(response))
print("reference total:", ref_logprobs.sum().item(), "nats over", len(gen_ids), "tokens")

' If you find a lost wallet, you should first check the contents to see if there is any identification or contact information. If there is, you should'
reference total: -15.74404239654541 nats over 30 tokens


## Step 1: tokenize jointly and identify the prompt/response boundary

We tokenize `prompt + response` as one string. To know where the response starts we tokenize the
prompt alone and check that its ids are a *prefix* of the joint ids. If they aren't, the boundary
merged and the score would be meaningless, so we refuse rather than guess.

In [3]:
full_ids = tokenizer(prompt + response, return_tensors="pt")["input_ids"][0]
prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"][0]
n_prompt = len(prompt_ids)
assert torch.equal(full_ids[:n_prompt], prompt_ids), "boundary merge!"
print("prompt tokens:", n_prompt, "| response tokens:", len(full_ids) - n_prompt)
print("boundary pieces:", [tokenizer.decode([i]) for i in full_ids[n_prompt-2:n_prompt+3]])
# And the joint response tokens are exactly the ones generate produced:
print("response ids match generated ids:", torch.equal(full_ids[n_prompt:], gen_ids.cpu()))

prompt tokens: 15 | response tokens: 30
boundary pieces: ['Assistant', ':', ' If', ' you', ' find']
response ids match generated ids: True


### The gotcha, demonstrated

Compare joint tokenization with "tokenize separately, then concatenate" for two versions of the same
response: with and without the leading space. Watch the token boundary at `Assistant:`.

In [4]:
def compare(prompt, response):
    joint = tokenizer(prompt + response)["input_ids"]
    concat = tokenizer(prompt)["input_ids"] + tokenizer(response)["input_ids"]
    same = joint == concat
    np_ = len(tokenizer(prompt)["input_ids"])
    print(f"response={response[:22]!r:26} joint==concat: {same!s:5}  "
          f"joint boundary: {[tokenizer.decode([i]) for i in joint[np_-1:np_+2]]}  "
          f"concat boundary: {[tokenizer.decode([i]) for i in concat[np_-1:np_+2]]}")

compare(prompt, response)              # leading space: fine
compare(prompt, response.lstrip())     # no leading space: 'Assistant:' + 'If' -> what happens?
compare(prompt + " ", response.lstrip())   # trailing space on the prompt: also a different boundary

response=' If you find a lost wa'   joint==concat: True   joint boundary: [':', ' If', ' you']  concat boundary: [':', ' If', ' you']
response='If you find a lost wal'   joint==concat: True   joint boundary: [':', 'If', ' you']  concat boundary: [':', 'If', ' you']
response='If you find a lost wal'   joint==concat: False  joint boundary: [' If', ' you', ' find']  concat boundary: [' ', 'If', ' you']


The first line is the clean case. The second shows that for *this* tokenizer a missing leading space
does **not** merge (`:` and `If` stay separate), so the score would be fine, though for a different
response than the model would naturally write (`If` vs ` If` are different tokens). The third line is
the real failure: with a trailing space on the prompt, the joint string tokenizes to `' If'` while the
concatenation gives `' '` then `'If'`, a sequence the model essentially never sees. Which cases merge
depends on the vocabulary, so never rely on it. The rule we adopt: **prompts end in `:` with no
trailing space, responses start with a space, and we always tokenize jointly and check the prefix.**

## Step 2: forward pass, log-softmax, gather the response tokens

The logits at position $t$ are the model's prediction for token $t+1$. So the response tokens at
positions `n_prompt ... T-1` are predicted by logits at positions `n_prompt-1 ... T-2`. Do the softmax
in fp32: the bf16 logits are fine to store, but summing 150k exponentials in bf16 is not.

In [5]:
with torch.no_grad():
    logits = model(full_ids[None].to(model.device)).logits[0]      # (T, vocab)
logprobs = torch.log_softmax(logits.float(), dim=-1)
targets = full_ids[n_prompt:].to(model.device)
token_lp = logprobs[n_prompt - 1 : -1].gather(1, targets[:, None])[:, 0]

total = token_lp.sum().item()
print(f"log P(response | prompt) = {total:.3f} nats  |  {total/len(targets):.3f} nats/token  |  {len(targets)} tokens")
print()
print(f"{'token':>16} {'teacher-forced':>15} {'from generate':>14}")
for tok, a, b in zip(targets.tolist(), token_lp.tolist(), ref_logprobs.tolist()):
    print(f"{tokenizer.decode([tok])!r:>16} {a:15.4f} {b:14.4f}")
print()
print("max |difference| vs generate's own scores:", (token_lp - ref_logprobs.to(token_lp.device)).abs().max().item())

log P(response | prompt) = -15.651 nats  |  -0.522 nats/token  |  30 tokens

           token  teacher-forced  from generate
           ' If'         -2.0701        -2.0618
          ' you'         -0.0166        -0.0166
         ' find'         -0.1443        -0.1443
            ' a'         -0.0060        -0.0059
         ' lost'         -0.0389        -0.0345
       ' wallet'         -0.0005        -0.0006
             ','         -0.0401        -0.0447
          ' you'         -1.3081        -1.2856
       ' should'         -0.2950        -0.2672
        ' first'         -1.1303        -1.1290
        ' check'         -1.3389        -1.3299
          ' the'         -1.4883        -1.4663
     ' contents'         -0.3412        -0.3366
           ' to'         -0.6872        -0.6862
          ' see'         -0.4167        -0.4288
           ' if'         -0.0157        -0.0157
        ' there'         -0.4259        -0.4393
           ' is'         -0.3227        -0.3231
          '

The two columns agree to bf16 round-off (the generation loop uses a KV cache and processes one token
at a time; the teacher-forced pass does the whole sequence at once, so tiny numerical differences are
expected). Greedy tokens have high probability, so per-token values are close to 0.

## Step 3: the packaged version

`persona_selection.scoring.score_response` is the code above with the checks built in. Later notebooks
import it. Confirm it gives the same number, and see what `add_eos=True` does: it appends
`<|endoftext|>` so the score also includes "and then the response ends". Whether Phase 1 should include
that term is a design choice: it matters when comparing responses of different lengths.

In [6]:
from persona_selection.scoring import score_response

s = score_response(model, tokenizer, prompt, response)
print(f"packaged: {s['logprob']:.3f} nats, {s['logprob_per_token']:.3f}/token  (hand: {total:.3f})")
s_eos = score_response(model, tokenizer, prompt, response, add_eos=True)
print(f"with eos: {s_eos['logprob']:.3f} nats; the eos token alone contributed {s_eos['token_logprobs'][-1]:.3f}")
print("  (a mid-sentence cut, so ending here is very unlikely; compare a complete answer below)")

complete = " If you find a lost wallet, you should try to return it to its owner."
s_c = score_response(model, tokenizer, prompt, complete, add_eos=True)
print(f"complete sentence + eos: {s_c['logprob']:.3f} nats over {s_c['n_tokens']} tokens; eos term {s_c['token_logprobs'][-1]:.3f}")

# The boundary check in action: the trailing-space prompt from the demo above. The prompt's last token
# is ' ' but the joint tokenization has ' If' there, so the prompt is no longer a prefix and we refuse.
try:
    score_response(model, tokenizer, prompt + " ", response.lstrip())
except ValueError as e:
    print("boundary check raised:", e)

packaged: -15.651 nats, -0.522/token  (hand: -15.651)
with eos: -31.539 nats; the eos token alone contributed -15.888
  (a mid-sentence cut, so ending here is very unlikely; compare a complete answer below)
complete sentence + eos: -12.793 nats over 18 tokens; eos term -3.484
boundary check raised: prompt tokens changed when joined with the response (boundary merge); use a prompt ending in ':' and a response starting with a space


## Sanity: a wrong response scores much lower

Same prompt, a fluent but off-topic response. The point is not the exact number but the scale: tens of
nats separate "plausible" from "implausible" for a 20-token response. Phase 1's mixture weights are
driven by *differences* of such numbers across persona labels, so keeping them in a sane range
(README 0.6) is what makes the fit informative rather than a hard argmax.

In [7]:
for r in [" If you find a lost wallet, you should try to return it to its owner.",
          " The mitochondria is the powerhouse of the cell.",
          " Keep the cash and throw the rest away."]:
    s = score_response(model, tokenizer, prompt, r)
    print(f"{s['logprob']:8.2f} nats  {s['logprob_per_token']:6.2f}/tok  ({s['n_tokens']:2d} tok)  {r!r}")

   -9.31 nats   -0.55/tok  (17 tok)  ' If you find a lost wallet, you should try to return it to its owner.'
  -23.59 nats   -2.36/tok  (10 tok)  ' The mitochondria is the powerhouse of the cell.'


  -26.94 nats   -2.99/tok  ( 9 tok)  ' Keep the cash and throw the rest away.'


In [8]:
(RESULTS / "0.4_scoring.json").write_text(json.dumps({
    "config": CONFIG, "prompt": prompt, "response": response,
    "teacher_forced_total": total, "generate_total": ref_logprobs.sum().item(),
    "token_logprobs": token_lp.tolist(), "tokens": [tokenizer.decode([t]) for t in targets.tolist()],
}, indent=2))
print("saved", RESULTS / "0.4_scoring.json")

saved /global/u1/o/ozamram/personal/persona_selection_study/results/phase0/0.4_scoring.json



## What we saw (Qwen2.5-7B base, 2026-09-21)

- Teacher-forced per-token log-probs matched `generate`'s own scores to within 0.08 nats per token
  (bf16 + KV-cache vs full-sequence pass). Total −15.7 nats over 30 greedy tokens, −0.52 nats/token.
- The `eos` term is large and length-dependent: −15.9 nats mid-sentence, −3.5 nats after a complete
  sentence. Including it in Phase 1 scores is a real modelling choice, not a detail.
- Scale check: a plausible on-topic answer scored ≈ −0.55 nats/token; an off-topic sentence ≈ −2.4;
  a fluent but "wrong-persona" answer ≈ −3.0. Differences of tens of nats between *responses* are
  normal; Phase 1 cares about differences between *labels* for the same response, which 0.6 shows are
  1–3 nats.
